# Parcels on an inflated surface (nilearn)

Projects the four group-level parcels (LO, pFS, pIPS, V1) onto `fsaverage` and plots
them in distinct colors, both hemispheres, lateral + medial.

All surface data caches into the cluster's `nilearn_data` (nothing downloads to your PC).
Written for nilearn 0.9–0.10; if you're on 0.11+ and `fetch_surf_fsaverage` breaks, tell me
and I'll switch to the new `SurfaceImage` API.

In [ ]:
# ============ CONFIG ============
import os
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
from nilearn import datasets, surface, plotting

ROI_DIR = '/user_data/csimmon2/git_repos/ptoc/roiParcels'

# edit filenames if they differ (case-sensitive). The load cell flags any that are missing.
PARCEL_FILES = {
    'LO':   os.path.join(ROI_DIR, 'LO.nii.gz'),
    'pFS':  os.path.join(ROI_DIR, 'PFS.nii.gz'),
    'pIPS': os.path.join(ROI_DIR, 'pIPS.nii.gz'),
    'V1':   os.path.join(ROI_DIR, 'V1.nii.gz'),
}

# If there is no combined pIPS parcel, merge Wang IPS0 + IPS1 once (then point PARCEL_FILES at it):
# from nilearn.image import math_img
# pips = math_img('(a>0)|(b>0)', a=nib.load(f'{ROI_DIR}/IPS0.nii.gz'), b=nib.load(f'{ROI_DIR}/IPS1.nii.gz'))
# pips.to_filename(f'{ROI_DIR}/pIPS.nii.gz')

THRESH = 0.0                 # binarize each parcel at value > THRESH
FSAVG  = 'fsaverage'         # 'fsaverage5' = fast/coarse ... 'fsaverage' = high-res/crisp

# fixed label value per parcel; COLORS are ordered by label value (1..4)
LABELS = {'LO': 1, 'pFS': 2, 'pIPS': 3, 'V1': 4}
COLORS = ['#009E73', '#E69F00', '#56B4E9', '#D55E00']   # LO, pFS, pIPS, V1  (Okabe-Ito, colorblind-safe)
PAINT_ORDER = ['V1', 'pIPS', 'pFS', 'LO']               # last painted wins in any voxel overlap

In [ ]:
# ============ LOAD + SANITY CHECK ============
# Parcels must be in MNI space (fsaverage is MNI). Expect shape (91,109,91), vox ~2mm.
# If any parcel prints a native/anat grid, projection will land in the wrong place.
imgs = {}
for name, path in PARCEL_FILES.items():
    if not os.path.exists(path):
        print(f'MISSING: {name:5s} -> {path}')
        continue
    img = nib.load(path)
    imgs[name] = img
    vox = np.round(np.sqrt((img.affine[:3, :3] ** 2).sum(0)), 2)
    nvox = int((img.get_fdata() > THRESH).sum())
    print(f'{name:5s} shape={img.shape} vox={vox} nvox(>{THRESH})={nvox}')

assert imgs, 'No parcels loaded — fix the paths above.'

In [ ]:
# ============ BUILD COMBINED LABEL VOLUME ============
from nilearn.image import resample_to_img

ref = next(iter(imgs.values()))                      # reference grid
label_data = np.zeros(ref.shape, dtype=np.int16)

for name in PAINT_ORDER:
    if name not in imgs:
        continue
    im = imgs[name]
    if im.shape != ref.shape or not np.allclose(im.affine, ref.affine):
        im = resample_to_img(im, ref, interpolation='nearest')
    label_data[im.get_fdata() > THRESH] = LABELS[name]

label_img = nib.Nifti1Image(label_data, ref.affine)
print('surface-label voxel counts:',
      {n: int((label_data == LABELS[n]).sum()) for n in LABELS})

In [ ]:
# ============ FETCH fsaverage + PROJECT TO SURFACE ============
# Caches to cluster nilearn_data on first run. nearest interpolation keeps labels discrete.
fsavg = datasets.fetch_surf_fsaverage(mesh=FSAVG)

tex = {}
for h in ['left', 'right']:
    tex[h] = surface.vol_to_surf(label_img, fsavg[f'pial_{h}'], interpolation='nearest')

In [ ]:
# ============ STATIC FIGURE (both hemis, lateral + medial) ============
cmap = ListedColormap(COLORS)
fig, axes = plt.subplots(2, 2, figsize=(11, 9), subplot_kw={'projection': '3d'})
panels = [('left', 'lateral', axes[0, 0]), ('right', 'lateral', axes[0, 1]),
          ('left', 'medial',  axes[1, 0]), ('right', 'medial',  axes[1, 1])]

for h, v, ax in panels:
    plotting.plot_surf_roi(
        fsavg[f'infl_{h}'], roi_map=tex[h], hemi=h, view=v,
        bg_map=fsavg[f'sulc_{h}'], bg_on_data=True,
        cmap=cmap, vmin=1, vmax=4, axes=ax, figure=fig,
    )
    ax.set_title(f'{h} {v}', fontsize=11)

handles = [mpatches.Patch(color=COLORS[LABELS[n] - 1], label=n) for n in ['LO', 'pFS', 'pIPS', 'V1']]
fig.legend(handles=handles, loc='lower center', ncol=4, frameon=False, fontsize=12)
fig.tight_layout(rect=[0, 0.04, 1, 1])
plt.show()

### Optional: interactive view
Rotatable single-hemisphere view for exploring. The colorbar renders continuous (expected for a label map);
the static figure above is the one to keep.

In [ ]:
plotting.view_surf(
    fsavg['infl_left'], tex['left'], cmap=cmap, vmin=1, vmax=4,
    bg_map=fsavg['sulc_left'], symmetric_cmap=False,
)